# EgoBody Interactive Missing Gaze Viewer

This notebook is for EgoBody only. It compares SparseGaze residual rollout with HAGI++ on missing/evaluation frames. Because the local EgoBody copy has no RGB, depth, or scene geometry, the notebook uses direction-level visualization only:

- missing-frame angular error timeline
- CPF pitch timeline
- CPF yaw timeline

It intentionally does not draw 3D gaze rays as a scanpath.


In [1]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import clear_output, display


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / 'Experiments').exists() and (base / 'src').exists():
            return base
    return Path('/home/liumu/Github_Projects/adt_dataset_sandbox')


REPO_ROOT = find_repo_root()
EGO_SCRIPT_PATH = REPO_ROOT / 'Experiments' / 'visualization & Analysis' / 'egobody' / '01_missing_gaze_direction_compare.py'

spec = importlib.util.spec_from_file_location('egobody_missing_gaze', EGO_SCRIPT_PATH)
ego = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = ego
spec.loader.exec_module(ego)

EVAL_ROOT = ego.DEFAULT_EVAL_ROOT
CACHE_ROOT = ego.DEFAULT_CACHE_ROOT
HAGI_DATA_PATH = ego.DEFAULT_HAGI_DATA_PATH
HAGI_OUTPUT_ROOT = ego.DEFAULT_HAGI_OUTPUT_ROOT
DEFAULT_SPARSEGAZE_REL = 'sparsegaze_cpf_forward_head_motion_residual_ss/test/rollout'


In [2]:
def sparsegaze_path(sequence: str, target_hz: int, phase: int, rel_dir: str = DEFAULT_SPARSEGAZE_REL) -> Path:
    return ego.prediction_path(EVAL_ROOT, rel_dir, sequence, target_hz, phase)


def available_target_hz(output_root: Path = HAGI_OUTPUT_ROOT) -> list[int]:
    values = set()
    for path in output_root.glob('sliding_primary_nsample20_framerate_*.npz'):
        try:
            values.add(int(path.stem.rsplit('_', 1)[-1]))
        except ValueError:
            pass
    return sorted(values, reverse=True) or [6]


def predicted_sequences(target_hz: int = 6, phase: int = 0) -> list[str]:
    root = EVAL_ROOT / DEFAULT_SPARSEGAZE_REL / 'sequence_predictions'
    if not root.exists():
        return []
    out = []
    for seq_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        if (seq_dir / f'hz{target_hz}_phase{phase}.npz').exists():
            cache_matches = list(CACHE_ROOT.glob(f'*/{seq_dir.name}.npz'))
            if cache_matches:
                out.append(seq_dir.name)
    return out


def load_comparison(sequence: str, target_hz: int, phase: int = 0):
    path = sparsegaze_path(sequence, target_hz, phase)
    if not path.exists():
        raise FileNotFoundError(f'Missing SparseGaze prediction: {path}')

    sparse_track, meta = ego.load_prediction_track(path, 'SparseGaze residual')
    n = len(meta['gt_xyz'])
    tracks = [sparse_track]

    hagi_record = None
    hagi_message = ''
    if HAGI_DATA_PATH.exists():
        try:
            hagi_record = ego.load_hagi_record(HAGI_DATA_PATH, sequence)
        except Exception as exc:
            hagi_message = f'HAGI++ data unavailable: {exc}'
    else:
        hagi_message = f'HAGI++ data missing: {HAGI_DATA_PATH}'

    hagi_path = ego.default_hagi_npz(HAGI_OUTPUT_ROOT, target_hz)
    if phase != 0:
        hagi_message = (hagi_message + ' ' if hagi_message else '') + 'HAGI++ low-framerate output is phase=0 only.'
    elif hagi_path.exists() and HAGI_DATA_PATH.exists():
        try:
            tracks.append(ego.load_hagi_track(hagi_path, HAGI_DATA_PATH, sequence, n))
        except Exception as exc:
            hagi_message = (hagi_message + ' ' if hagi_message else '') + f'HAGI++ prediction unavailable: {exc}'
    else:
        hagi_message = (hagi_message + ' ' if hagi_message else '') + f'HAGI++ file missing: {hagi_path}'

    errors = {}
    for track in tracks:
        if track.angular_error_deg is None:
            errors[track.label] = ego.angular_error_deg(track.pred_xyz[:n], meta['gt_xyz'])
        else:
            full = np.full(n, np.nan, dtype=float)
            count = min(n, len(track.angular_error_deg))
            full[:count] = track.angular_error_deg[:count]
            errors[track.label] = full

    return meta, hagi_record, tracks, errors, hagi_message


def summarize_window(meta, tracks, errors, rows_mask):
    rows = []
    for track in tracks:
        mask = rows_mask & meta['eval_mask'] & np.isfinite(errors[track.label])
        values = errors[track.label][mask]
        rows.append({
            'track': track.label,
            'valid_eval_frames': int(mask.sum()),
            'mean_error_deg': float(np.nanmean(values)) if len(values) else np.nan,
            'median_error_deg': float(np.nanmedian(values)) if len(values) else np.nan,
            'p90_error_deg': float(np.nanpercentile(values, 90)) if len(values) else np.nan,
        })
    return pd.DataFrame(rows)


In [3]:
TRACK_COLORS = {
    'SparseGaze residual': '#ff7f0e',
    'HAGI++': '#2ca02c',
    'GT': '#111111',
    'anchors': '#777777',
}


def add_sparse_line(fig, *, x, y, name, color, row, col=1, mode='lines+markers', width=1.6, marker_size=4):
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode=mode,
            name=name,
            marker={'size': marker_size, 'color': color},
            line={'width': width, 'color': color},
            hovertemplate='frame=%{x}<br>value=%{y:.2f}<extra></extra>',
        ),
        row=row,
        col=col,
    )


def build_view(
    sequence: str,
    target_hz: int,
    phase: int,
    start_frame: int,
    end_frame: int,
    focus_frame: int,
    show_anchors: bool,
    show_gt: bool,
    show_sparsegaze: bool,
    show_hagi: bool,
):
    meta, hagi_record, tracks, errors, hagi_message = load_comparison(sequence, target_hz, phase)
    n = len(meta['gt_xyz'])

    start_frame = max(0, min(int(start_frame), n - 1))
    end_frame = max(start_frame + 1, min(int(end_frame), n))
    focus_frame = max(start_frame, min(int(focus_frame), end_frame - 1))

    rows_mask = ego.frame_window_mask(n, start_frame, end_frame)
    eval_mask = rows_mask & meta['eval_mask']
    frames = np.arange(n)

    enabled = set()
    if show_sparsegaze:
        enabled.add('SparseGaze residual')
    if show_hagi:
        enabled.add('HAGI++')

    subplot_titles = [
        f'Angular error | frames {start_frame}-{end_frame - 1}',
        'CPF pitch | GT and missing-frame predictions',
        'CPF yaw | GT and missing-frame predictions',
    ]
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=subplot_titles,
    )

    for track in tracks:
        if track.label not in enabled:
            continue
        y = np.full(n, np.nan, dtype=float)
        mask = eval_mask & np.isfinite(errors[track.label])
        y[mask] = errors[track.label][mask]
        add_sparse_line(
            fig,
            x=frames,
            y=y,
            name=f'{track.label} error',
            color=TRACK_COLORS.get(track.label),
            row=1,
        )

    if show_anchors:
        anchor_x = frames[rows_mask & meta['anchor_mask']]
        fig.add_trace(
            go.Scatter(
                x=anchor_x,
                y=np.zeros_like(anchor_x),
                mode='markers',
                name='anchors',
                marker={'size': 4, 'color': TRACK_COLORS['anchors'], 'opacity': 0.5},
                hovertemplate='anchor frame=%{x}<extra></extra>',
            ),
            row=1,
            col=1,
        )

    if hagi_record is None:
        hagi_message = (hagi_message + ' ' if hagi_message else '') + 'CPF pitch/yaw unavailable without HAGI++ T_world_CPF.'
    else:
        n_cpf = min(n, len(hagi_record['T_world_CPF']))
        cpf_rows_mask = rows_mask[:n_cpf]
        cpf_eval_mask = eval_mask[:n_cpf]
        cpf_anchor_mask = meta['anchor_mask'][:n_cpf]
        cpf_frames = frames[:n_cpf]
        gt_py = ego.cpf_to_pitch_yaw_deg(ego.world_to_hagi_cpf(meta['gt_xyz'], hagi_record, n_cpf))

        if show_gt:
            for row, dim, axis_name in [(2, 0, 'pitch'), (3, 1, 'yaw')]:
                mask = cpf_rows_mask & np.isfinite(gt_py[:, dim])
                add_sparse_line(
                    fig,
                    x=cpf_frames[mask],
                    y=gt_py[mask, dim],
                    name=f'GT {axis_name}',
                    color=TRACK_COLORS['GT'],
                    row=row,
                    mode='lines',
                    width=2.1,
                    marker_size=0,
                )
                if show_anchors:
                    anchor_mask = cpf_rows_mask & cpf_anchor_mask & np.isfinite(gt_py[:, dim])
                    fig.add_trace(
                        go.Scatter(
                            x=cpf_frames[anchor_mask],
                            y=gt_py[anchor_mask, dim],
                            mode='markers',
                            name=f'anchor GT {axis_name}',
                            marker={'size': 4, 'color': TRACK_COLORS['anchors'], 'opacity': 0.45},
                            hovertemplate='anchor frame=%{x}<br>value=%{y:.2f}<extra></extra>',
                        ),
                        row=row,
                        col=1,
                    )

        for track in tracks:
            if track.label not in enabled:
                continue
            pred_py = ego.cpf_to_pitch_yaw_deg(ego.world_to_hagi_cpf(track.pred_xyz, hagi_record, n_cpf))
            valid = cpf_eval_mask & np.isfinite(errors[track.label][:n_cpf])
            for row, dim, axis_name in [(2, 0, 'pitch'), (3, 1, 'yaw')]:
                mask = valid & np.isfinite(pred_py[:, dim])
                add_sparse_line(
                    fig,
                    x=cpf_frames[mask],
                    y=pred_py[mask, dim],
                    name=f'{track.label} {axis_name}',
                    color=TRACK_COLORS.get(track.label),
                    row=row,
                    width=1.4,
                    marker_size=3,
                )

    for row in [1, 2, 3]:
        fig.add_vline(x=focus_frame, line_width=1, line_dash='dash', line_color='#666666', row=row, col=1)

    fig.update_xaxes(title='Frame index', range=[start_frame, end_frame - 1], row=3, col=1)
    fig.update_yaxes(title='Angular error [deg]', rangemode='tozero', row=1, col=1)
    fig.update_yaxes(title='Pitch [deg]', row=2, col=1)
    fig.update_yaxes(title='Yaw [deg]', row=3, col=1)
    fig.update_layout(
        height=900,
        width=1280,
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': -0.16, 'xanchor': 'left', 'x': 0.0},
        margin={'l': 60, 'r': 20, 't': 80, 'b': 110},
    )

    summary = summarize_window(meta, tracks, errors, rows_mask)
    return fig, summary, hagi_message, n


In [4]:
hz_values = available_target_hz()
default_hz = 6 if 6 in hz_values else hz_values[0]
sequences = predicted_sequences(default_hz, phase=0)
if not sequences:
    raise RuntimeError(f'No EgoBody SparseGaze predictions found under {EVAL_ROOT / DEFAULT_SPARSEGAZE_REL}')

default_sequence = 'recording_20210907_S03_S04_01'
if default_sequence not in sequences:
    default_sequence = sequences[0]

sequence_dropdown = widgets.Dropdown(options=sequences, value=default_sequence, description='sequence', layout=widgets.Layout(width='640px'))
hz_dropdown = widgets.Dropdown(options=hz_values, value=default_hz, description='Hz')
phase_text = widgets.BoundedIntText(value=0, min=0, max=4, step=1, description='phase', layout=widgets.Layout(width='150px'))
start_text = widgets.BoundedIntText(value=149, min=0, max=10_000, step=1, description='start', layout=widgets.Layout(width='170px'))
end_text = widgets.BoundedIntText(value=300, min=1, max=10_000, step=1, description='end', layout=widgets.Layout(width='170px'))
focus_slider = widgets.IntSlider(value=299, min=149, max=299, step=1, description='focus', continuous_update=False, layout=widgets.Layout(width='520px'))
show_anchors_checkbox = widgets.Checkbox(value=True, description='anchors')
show_gt_checkbox = widgets.Checkbox(value=True, description='GT')
show_sparse_checkbox = widgets.Checkbox(value=True, description='SparseGaze')
show_hagi_checkbox = widgets.Checkbox(value=True, description='HAGI++')
build_button = widgets.Button(description='Build view', button_style='primary', icon='refresh')
output = widgets.Output()


def refresh_sequences(*_):
    seqs = predicted_sequences(int(hz_dropdown.value), int(phase_text.value))
    if not seqs:
        sequence_dropdown.options = []
        return
    old = sequence_dropdown.value
    sequence_dropdown.options = seqs
    sequence_dropdown.value = old if old in seqs else seqs[0]


def sync_focus_bounds(*_):
    start = int(start_text.value)
    end = max(start + 1, int(end_text.value))
    end_text.value = end
    focus_slider.min = start
    focus_slider.max = end - 1
    if focus_slider.value < start or focus_slider.value >= end:
        focus_slider.value = end - 1


hz_dropdown.observe(refresh_sequences, names='value')
phase_text.observe(refresh_sequences, names='value')
start_text.observe(sync_focus_bounds, names='value')
end_text.observe(sync_focus_bounds, names='value')
sync_focus_bounds()


def on_build(_=None):
    with output:
        clear_output(wait=True)
        if not sequence_dropdown.options:
            print('No sequence has predictions for this Hz/phase.')
            return
        try:
            fig, summary, hagi_message, n_frames = build_view(
                sequence=sequence_dropdown.value,
                target_hz=int(hz_dropdown.value),
                phase=int(phase_text.value),
                start_frame=int(start_text.value),
                end_frame=int(end_text.value),
                focus_frame=int(focus_slider.value),
                show_anchors=bool(show_anchors_checkbox.value),
                show_gt=bool(show_gt_checkbox.value),
                show_sparsegaze=bool(show_sparse_checkbox.value),
                show_hagi=bool(show_hagi_checkbox.value),
            )
            print(f'sequence={sequence_dropdown.value} | frames={n_frames} | Hz={hz_dropdown.value} | phase={phase_text.value}')
            if hagi_message:
                print(hagi_message)
            display(summary)
            display(fig)
        except Exception as exc:
            print(f'{type(exc).__name__}: {exc}')


build_button.on_click(on_build)
controls = widgets.VBox([
    widgets.HBox([sequence_dropdown]),
    widgets.HBox([hz_dropdown, phase_text, start_text, end_text]),
    focus_slider,
    widgets.HBox([show_anchors_checkbox, show_gt_checkbox, show_sparse_checkbox, show_hagi_checkbox, build_button]),
])
display(controls, output)
on_build()


Output()

Notes:

- This notebook is direction-level only. It does not draw 3D rays or scanpath endpoints because the local EgoBody copy has no RGB/depth/scene-hit point.
- The CPF pitch/yaw panel uses the EgoBody/HAGI++ convention `cpf = -R_world_CPF.T @ world_direction`, matching `01_missing_gaze_direction_compare.py`.
- HAGI++ low-framerate outputs currently use phase 0 and start after the sliding history window, so early ranges may show no HAGI++ points.
- The error curve is plotted only on missing/evaluation frames. Anchor frames are shown at zero only as reference markers.
